# DATA 621 - Homework 02: Classification Metrics

## Project objective

This project evaluates an already scored binary classifier using the supplied actual labels, predicted labels, and predicted probabilities. The goal is to calculate and interpret threshold-based classification metrics and probability-based ROC/AUC, then verify the calculations with `scikit-learn`. Model training is outside the scope of this assignment because the provided data already contains the model's outputs. Class `1` is treated as the positive class throughout.


## Data Loading, Inspection and Confusion Matrix

### Data Loading and Inspection
To complete this assignment, the dataset was first uploaded to the group GitHub repository and then loaded directly from the raw GitHub URL using `pandas`. By using `pandas.read_csv()` with the raw GitHub URL, it ensures the notebook is 100% reproducible without requiring local file downloads.   

In [11]:
import pandas as pd
import numpy as np

# Load the dataset directly from the raw GitHub URL
url = "https://raw.githubusercontent.com/alinsimon/data621-data-mining/refs/heads/main/homeworks/hw02/data/classification-output-data.csv"
df = pd.read_csv(url)

# Verify dataset dimensions (181 observations, 11 columns)
print(f"Dataset shape: {df.shape}")

# Inspect the first 5 rows of the three key columns
key_cols = ['class', 'scored.class', 'scored.probability']
print(df[key_cols].head())

Dataset shape: (181, 11)
   class  scored.class  scored.probability
0      0             0            0.328452
1      0             0            0.273190
2      1             0            0.109660
3      0             0            0.055998
4      0             0            0.100491


### Confusion Matrix
The cleanest way to generate a labeled raw confusion matrix is by using `pd.crosstab()` because the order in which the arguments are passed explicitly defines what the rows and columns represent.

In [12]:
def get_confusion_matrix(data, actual_col='class', predicted_col='scored.class'):
    """
    Returns the raw confusion matrix using pandas crosstab.
    Rows (index) = Actual Class
    Columns = Predicted Class
    """
    return pd.crosstab(
        index=data[actual_col],
        columns=data[predicted_col],
        rownames=['Actual (class)'],
        colnames=['Predicted (scored.class)'],
        margins=True  # Adds row/column totals for easy verification
    )

conf_matrix = get_confusion_matrix(df)
print(conf_matrix)

Predicted (scored.class)    0   1  All
Actual (class)                        
0                         119   5  124
1                          30  27   57
All                       149  32  181


#### Confusion Matrix Explained

In the primary `pd.crosstab(df['class'], df['scored.class']` output, the rows represent the actual class (`class`), while the columns represent the predicted class (`scored.class`) at the 0.5 probability threshold. 
Row `0` contains all 124 observations that are actually negative (`class == 0`) and Row `1` contains all 57 observations that are actually positive (`class == 1`).
Column `0` contains all 149 observations predicted to be negative (`scored.class == 0`), and Column `1` contains all 32 observations predicted to be positive (`scored.class == 1`).

*Breakdown of Quadrants (N = 181)*

| Metric | Cell Coordinate (Actual, Predicted) | Count | Percentage | Meaning |
|:---|:---|---:|---:|:---|
| **True Negatives ($TN$)** | `(0, 0)` — Top-Left | **119** | 65.75% | The actual class was `0` and the model correctly predicted `0`. |
| **False Positives ($FP$)** | `(0, 1)` — Top-Right | **5** | 2.76% | The actual class was `0`, but the model incorrectly predicted `1` (Type I Error). |
| **False Negatives ($FN$)** | `(1, 0)` — Bottom-Left | **30** | 16.57% | The actual class was `1`, but the model incorrectly predicted `0` (Type II Error). |
| **True Positives ($TP$)** | `(1, 1)` — Bottom-Right | **27** | 14.92% | The actual class was `1` and the model correctly predicted `1`. |

Out of 181 total observations, the model classified $119 + 27 = 146$ correctly ($80.66\%$ accuracy) and misclassified $5 + 30 = 35$ observations ($19.34\%$ error rate).The errors are heavily skewed toward False Negatives (30) compared to False Positives (5), meaning the model is conservative at the 0.5 threshold—when it predicts a positive case (1), it is usually right, but it misses more than half of the actual positive cases ($27 / 57 \approx 47.37\%$ sensitivity).